# Multimodal Pet Adoption Speed Prediction

In [ ]:
%pip -q install sentence-transformers

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import json
import numpy as np
import torch
from pathlib import Path

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
PROJECT_DIR = Path("/content/drive/MyDrive/PetAdoption_Project")
MODEL_DIR = PROJECT_DIR / "models"
FEATURE_DIR = PROJECT_DIR / "features"
for directory in [MODEL_DIR, FEATURE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)
print("Device:", device)


## Data loading


In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path("/content/drive/MyDrive/PetAdoption_Project")
DATA_DIR = PROJECT_DIR / "data"

df = pd.read_csv(DATA_DIR / "train.csv")
breeds = pd.read_csv(DATA_DIR / "BreedLabels.csv")
colors = pd.read_csv(DATA_DIR / "ColorLabels.csv")
states = pd.read_csv(DATA_DIR / "StateLabels.csv")

print("Dataset shape:", df.shape)
print("Duplicate PetIDs:", df["PetID"].duplicated().sum())

missing_description = (
    df["Description"].fillna("").astype(str).str.strip().eq("")
)

print("Missing or empty descriptions:", missing_description.sum())
print("\nClass distribution:")
print(df["AdoptionSpeed"].value_counts().sort_index())

display(df.head(3))

## Data quality


In [ ]:
OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

quality_summary = pd.DataFrame({
    "Missing": df.isna().sum(),
    "MissingPercent": df.isna().mean().mul(100).round(2),
    "UniqueValues": df.nunique()
})

class_summary = (
    df["AdoptionSpeed"]
    .value_counts()
    .sort_index()
    .rename_axis("Class")
    .to_frame("Count")
)

class_summary["Percent"] = (
    class_summary["Count"] / len(df) * 100
).round(2)

description_lengths = (
    df["Description"].fillna("").astype(str).str.split().str.len()
)

print("Data quality summary:")
display(quality_summary)

print("Class distribution:")
display(class_summary)

print("Description length in words:")
display(description_lengths.describe().to_frame("WordCount"))

print("Records with PhotoAmt greater than zero:", df["PhotoAmt"].gt(0).sum())
print("Records with no listed photos:", df["PhotoAmt"].eq(0).sum())
print(
    "Records with descriptions and listed photos:",
    ((~missing_description) & df["PhotoAmt"].gt(0)).sum()
)

quality_summary.to_csv(OUTPUT_DIR / "data_quality_summary.csv")
class_summary.to_csv(OUTPUT_DIR / "class_distribution.csv")

print("Summary files saved.")

## Image matching


In [ ]:
import zipfile
from pathlib import PurePosixPath

ZIP_PATH = DATA_DIR / "train_images.zip"

eligible_df = df.loc[
    (~missing_description) & df["PhotoAmt"].gt(0)
].copy()

with zipfile.ZipFile(ZIP_PATH, "r") as archive:
    image_members = [
        name for name in archive.namelist()
        if not name.endswith("/")
        and name.lower().endswith((".jpg", ".jpeg", ".png"))
        and "__MACOSX" not in name
    ]

image_lookup = {}

for member in image_members:
    filename = PurePosixPath(member).name
    image_lookup[filename] = member

eligible_df["ImageMember"] = eligible_df["PetID"].map(
    lambda pet_id: image_lookup.get(f"{pet_id}-1.jpg")
)

print("Archive size (GB):", round(ZIP_PATH.stat().st_size / 1e9, 2))
print("Image files in archive:", len(image_members))
print("Eligible records:", len(eligible_df))
print("First images found:", eligible_df["ImageMember"].notna().sum())
print("First images missing:", eligible_df["ImageMember"].isna().sum())

print("\nExample archive paths:")
for member in image_members[:5]:
    print(member)

print("\nExample matched records:")
display(eligible_df[["PetID", "ImageMember", "AdoptionSpeed"]].head())

## Image extraction and validation


In [ ]:
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
import shutil

IMAGE_DIR = Path("/content/pet_adoption_images")
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

valid_records = []
invalid_records = []

with zipfile.ZipFile(ZIP_PATH, "r") as archive:
    for pet_id, member in tqdm(
        eligible_df[["PetID", "ImageMember"]].itertuples(
            index=False, name=None
        ),
        total=len(eligible_df),
        desc="Extracting and validating images"
    ):
        image_path = IMAGE_DIR / f"{pet_id}-1.jpg"

        try:
            if not image_path.exists():
                with archive.open(member) as source:
                    with image_path.open("wb") as destination:
                        shutil.copyfileobj(source, destination)

            with Image.open(image_path) as image:
                image.convert("RGB").load()

            valid_records.append({
                "PetID": pet_id,
                "ImageFile": image_path.name
            })

        except Exception as error:
            invalid_records.append({
                "PetID": pet_id,
                "Error": str(error)
            })

valid_images = pd.DataFrame(
    valid_records,
    columns=["PetID", "ImageFile"]
)

invalid_images = pd.DataFrame(
    invalid_records,
    columns=["PetID", "Error"]
)

model_df = eligible_df.merge(
    valid_images,
    on="PetID",
    how="inner",
    validate="one_to_one"
).reset_index(drop=True)

model_df.to_csv(DATA_DIR / "model_data.csv", index=False)
invalid_images.to_csv(
    OUTPUT_DIR / "invalid_images.csv",
    index=False
)

print("Valid images:", len(valid_images))
print("Invalid images:", len(invalid_images))
print("Final dataset shape:", model_df.shape)
print("Dataset saved:", DATA_DIR / "model_data.csv")

if not invalid_images.empty:
    display(invalid_images.head())

## Fixed data splits


In [ ]:
SPLIT_DIR = PROJECT_DIR / "splits"
SPLIT_DIR.mkdir(parents=True, exist_ok=True)
if all((SPLIT_DIR / name).exists() for name in ["train.csv", "validation.csv", "test.csv"]):
    train_df = pd.read_csv(SPLIT_DIR / "train.csv")
    val_df = pd.read_csv(SPLIT_DIR / "validation.csv")
    test_df = pd.read_csv(SPLIT_DIR / "test.csv")
    assert set(train_df["PetID"]).isdisjoint(val_df["PetID"])
    assert set(train_df["PetID"]).isdisjoint(test_df["PetID"])
    assert set(val_df["PetID"]).isdisjoint(test_df["PetID"])
    assert set(pd.concat([train_df, val_df, test_df])["PetID"]) == set(model_df["PetID"])
    print("Existing splits loaded.")
else:
    from sklearn.model_selection import train_test_split

    SEED = 42
    SPLIT_DIR = PROJECT_DIR / "splits"
    SPLIT_DIR.mkdir(parents=True, exist_ok=True)

    train_df, remaining_df = train_test_split(
        model_df,
        test_size=0.30,
        stratify=model_df["AdoptionSpeed"],
        random_state=SEED
    )

    val_df, test_df = train_test_split(
        remaining_df,
        test_size=0.50,
        stratify=remaining_df["AdoptionSpeed"],
        random_state=SEED
    )

    train_df = train_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)

    train_ids = set(train_df["PetID"])
    val_ids = set(val_df["PetID"])
    test_ids = set(test_df["PetID"])

    assert train_ids.isdisjoint(val_ids)
    assert train_ids.isdisjoint(test_ids)
    assert val_ids.isdisjoint(test_ids)
    assert len(train_ids | val_ids | test_ids) == len(model_df)

    train_df.to_csv(SPLIT_DIR / "train.csv", index=False)
    val_df.to_csv(SPLIT_DIR / "validation.csv", index=False)
    test_df.to_csv(SPLIT_DIR / "test.csv", index=False)

    split_summary = pd.DataFrame({
        "Train": train_df["AdoptionSpeed"].value_counts(),
        "Validation": val_df["AdoptionSpeed"].value_counts(),
        "Test": test_df["AdoptionSpeed"].value_counts()
    }).sort_index().astype(int)

    split_summary.index.name = "Class"
    split_summary.to_csv(OUTPUT_DIR / "split_class_distribution.csv")

    print("Training records:", len(train_df))
    print("Validation records:", len(val_df))
    print("Test records:", len(test_df))
    print("PetID overlap: 0")

    display(split_summary)

    print("Split files saved:", SPLIT_DIR)


## Tabular preprocessing and evaluation metrics


In [ ]:
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, cohen_kappa_score

numeric_columns = [
    "Age", "Quantity", "Fee", "VideoAmt", "PhotoAmt"
]

categorical_columns = [
    "Type", "Breed1", "Breed2", "Gender",
    "Color1", "Color2", "Color3",
    "MaturitySize", "FurLength",
    "Vaccinated", "Dewormed", "Sterilized",
    "Health", "State"
]

tabular_columns = numeric_columns + categorical_columns

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

tabular_preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_columns),
    ("categorical", categorical_pipeline, categorical_columns)
])

tabular_model = Pipeline([
    ("preprocessor", tabular_preprocessor),
    ("classifier", LogisticRegression(
        C=1.0,
        class_weight="balanced",
        max_iter=3000,
        random_state=SEED
    ))
])

def evaluate_predictions(y_true, y_pred):
    return {
        "MacroF1": f1_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "WeightedF1": f1_score(
            y_true, y_pred, average="weighted", zero_division=0
        ),
        "QWK": cohen_kappa_score(
            y_true, y_pred, weights="quadratic"
        )
    }



## Tabular model selection


In [ ]:
from sklearn.base import clone

search_results = []
best_score = float("-inf")
best_tabular_model = None

for c_value in [0.1, 1.0, 10.0]:
    for class_weight in [None, "balanced"]:
        candidate = clone(tabular_model)

        candidate.set_params(
            classifier__C=c_value,
            classifier__class_weight=class_weight
        )

        candidate.fit(
            train_df[tabular_columns],
            train_df["AdoptionSpeed"]
        )

        train_predictions = candidate.predict(
            train_df[tabular_columns]
        )

        val_predictions = candidate.predict(
            val_df[tabular_columns]
        )

        train_scores = evaluate_predictions(
            train_df["AdoptionSpeed"],
            train_predictions
        )

        val_scores = evaluate_predictions(
            val_df["AdoptionSpeed"],
            val_predictions
        )

        search_results.append({
            "C": c_value,
            "ClassWeight": (
                "None" if class_weight is None else class_weight
            ),
            "TrainMacroF1": train_scores["MacroF1"],
            "ValidationMacroF1": val_scores["MacroF1"],
            "ValidationWeightedF1": val_scores["WeightedF1"],
            "ValidationQWK": val_scores["QWK"]
        })

        if val_scores["MacroF1"] > best_score:
            best_score = val_scores["MacroF1"]
            best_tabular_model = candidate

tabular_search_results = pd.DataFrame(search_results).sort_values(
    "ValidationMacroF1",
    ascending=False
).reset_index(drop=True)

tabular_model = best_tabular_model

joblib.dump(
    tabular_model,
    MODEL_DIR / "tabular_best.joblib"
)

tabular_search_results.to_csv(
    OUTPUT_DIR / "tabular_parameter_search.csv",
    index=False
)

display(tabular_search_results.round(4))
print("Best tabular model saved.")

## Frozen multilingual text encoder


In [ ]:
import torch
from sentence_transformers import SentenceTransformer

torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"

gpu_check = torch.ones((2, 2), device=device)
print("Computation check:", (gpu_check @ gpu_check).cpu().tolist())

TEXT_MODEL_NAME = (
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

text_encoder = SentenceTransformer(
    TEXT_MODEL_NAME,
    device=device
)

text_encoder.max_seq_length = 256

sample_features = text_encoder.encode(
    [
        "A friendly cat looking for a loving home.",
        "A playful dog ready for adoption."
    ],
    batch_size=2,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False
)

print("Text model:", TEXT_MODEL_NAME)
print("Device:", device)
print("Maximum sequence length:", text_encoder.max_seq_length)
print("Sample feature shape:", sample_features.shape)

## Text feature extraction


In [ ]:
cached_splits = [("train", train_df), ("validation", val_df), ("test", test_df)]
cache_ready = all((FEATURE_DIR / f"text_{name}.npz").exists() for name, frame in cached_splits)
if cache_ready:
    for name, frame in cached_splits:
        with np.load(FEATURE_DIR / f"text_{name}.npz", allow_pickle=False) as saved:
            assert np.array_equal(saved["pet_ids"], frame["PetID"].astype(str).to_numpy())
            assert np.isfinite(saved["embeddings"]).all()
    print("Existing text features validated and reused.")
else:
    import json
    import numpy as np

    text_features = {}

    for split_name, split_df in [
        ("train", train_df),
        ("validation", val_df),
        ("test", test_df)
    ]:
        texts = (
            split_df["Description"]
            .fillna("")
            .astype(str)
            .tolist()
        )

        embeddings = text_encoder.encode(
            texts,
            batch_size=64,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=True
        ).astype(np.float32)

        assert embeddings.shape == (len(split_df), 384)
        assert np.isfinite(embeddings).all()

        text_features[split_name] = embeddings

        np.savez_compressed(
            FEATURE_DIR / f"text_{split_name}.npz",
            pet_ids=np.asarray(split_df["PetID"].astype(str).tolist(), dtype=str),
            embeddings=embeddings
        )

        print(f"{split_name}: {embeddings.shape}")
        print(f"Saved: text_{split_name}.npz")

    text_config = {
        "model_name": TEXT_MODEL_NAME,
        "max_sequence_length": text_encoder.max_seq_length,
        "normalize_embeddings": True,
        "embedding_dimension": 384,
        "batch_size": 64,
        "encoder_frozen": True
    }

    with (FEATURE_DIR / "text_config.json").open(
        "w", encoding="utf-8"
    ) as file:
        json.dump(text_config, file, indent=2)

    print("Text feature extraction completed.")


## Text model selection


In [ ]:
import joblib
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, cohen_kappa_score

loaded_text_features = {}

for split_name, split_df in [
    ("train", train_df),
    ("validation", val_df)
]:
    with np.load(
        FEATURE_DIR / f"text_{split_name}.npz",
        allow_pickle=False
    ) as saved:
        assert np.array_equal(
            saved["pet_ids"],
            split_df["PetID"].astype(str).to_numpy()
        )
        loaded_text_features[split_name] = saved["embeddings"].copy()

X_text_train = loaded_text_features["train"]
X_text_val = loaded_text_features["validation"]

y_train = train_df["AdoptionSpeed"].to_numpy()
y_val = val_df["AdoptionSpeed"].to_numpy()

text_search_rows = []
best_text_score = float("-inf")
text_model = None

for c_value in [0.1, 1.0, 10.0]:
    for class_weight in [None, "balanced"]:
        candidate = LogisticRegression(
            C=c_value,
            class_weight=class_weight,
            max_iter=3000,
            random_state=SEED
        )

        candidate.fit(X_text_train, y_train)

        train_scores = evaluate_predictions(
            y_train,
            candidate.predict(X_text_train)
        )

        val_scores = evaluate_predictions(
            y_val,
            candidate.predict(X_text_val)
        )

        text_search_rows.append({
            "C": c_value,
            "ClassWeight": (
                "None" if class_weight is None else class_weight
            ),
            "TrainMacroF1": train_scores["MacroF1"],
            "ValidationMacroF1": val_scores["MacroF1"],
            "ValidationWeightedF1": val_scores["WeightedF1"],
            "ValidationQWK": val_scores["QWK"]
        })

        if val_scores["MacroF1"] > best_text_score:
            best_text_score = val_scores["MacroF1"]
            text_model = candidate

text_search_results = pd.DataFrame(
    text_search_rows
).sort_values(
    "ValidationMacroF1",
    ascending=False
).reset_index(drop=True)

joblib.dump(text_model, MODEL_DIR / "text_best.joblib")

text_search_results.to_csv(
    OUTPUT_DIR / "text_parameter_search.csv",
    index=False
)

display(text_search_results.round(4))
print("Best text model saved.")

## Frozen image encoder


In [ ]:
from pathlib import Path
import torch
import torchvision
from torchvision.models import resnet18, ResNet18_Weights

IMAGE_DIR = Path("/content/pet_adoption_images")

print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)
print("GPU available:", torch.cuda.is_available())
print("Local image count:", len(list(IMAGE_DIR.glob("*.jpg"))))

image_weights = ResNet18_Weights.DEFAULT
image_transform = image_weights.transforms()

image_encoder = resnet18(weights=image_weights)
image_encoder.fc = torch.nn.Identity()

for parameter in image_encoder.parameters():
    parameter.requires_grad = False

image_encoder = image_encoder.to(device)
image_encoder.eval()

with torch.inference_mode():
    sample_image_features = image_encoder(
        torch.zeros((2, 3, 224, 224), device=device)
    )

print("Image encoder: ResNet18")
print("Feature shape:", tuple(sample_image_features.shape))

## Image feature extraction


In [ ]:
cached_splits = [("train", train_df), ("validation", val_df), ("test", test_df)]
cache_ready = all((FEATURE_DIR / f"image_{name}.npz").exists() for name, frame in cached_splits)
if cache_ready:
    for name, frame in cached_splits:
        with np.load(FEATURE_DIR / f"image_{name}.npz", allow_pickle=False) as saved:
            assert np.array_equal(saved["pet_ids"], frame["PetID"].astype(str).to_numpy())
            assert np.isfinite(saved["embeddings"]).all()
    print("Existing image features validated and reused.")
else:
    import json
    import numpy as np
    import torch
    from torch.utils.data import Dataset, DataLoader
    from PIL import Image
    from tqdm.auto import tqdm

    class PetImageDataset(Dataset):
        def __init__(self, dataframe, image_dir, transform):
            self.pet_ids = dataframe["PetID"].astype(str).tolist()
            self.image_dir = image_dir
            self.transform = transform

        def __len__(self):
            return len(self.pet_ids)

        def __getitem__(self, index):
            image_path = self.image_dir / f"{self.pet_ids[index]}-1.jpg"

            with Image.open(image_path) as image:
                image = image.convert("RGB")
                tensor = self.transform(image)

            return tensor

    image_encoder.eval()

    for split_name, split_df in [
        ("train", train_df),
        ("validation", val_df),
        ("test", test_df)
    ]:
        dataset = PetImageDataset(
            split_df,
            IMAGE_DIR,
            image_transform
        )

        loader = DataLoader(
            dataset,
            batch_size=64,
            shuffle=False,
            num_workers=2,
            pin_memory=True
        )

        feature_batches = []

        with torch.inference_mode():
            for images in tqdm(
                loader,
                desc=f"Encoding {split_name}"
            ):
                images = images.to(device, non_blocking=True)
                features = image_encoder(images)
                feature_batches.append(features.cpu().numpy())

        embeddings = np.concatenate(
            feature_batches,
            axis=0
        ).astype(np.float32)

        assert embeddings.shape == (len(split_df), 512)
        assert np.isfinite(embeddings).all()

        np.savez_compressed(
            FEATURE_DIR / f"image_{split_name}.npz",
            pet_ids=np.asarray(
                split_df["PetID"].astype(str).tolist(),
                dtype=str
            ),
            embeddings=embeddings
        )

        print(f"{split_name}: {embeddings.shape}")
        print(f"Saved: image_{split_name}.npz")

    image_config = {
        "architecture": "resnet18",
        "weights": str(image_weights),
        "transform": str(image_transform),
        "embedding_dimension": 512,
        "batch_size": 64,
        "encoder_frozen": True,
        "image_selection": "first image per pet"
    }

    with (FEATURE_DIR / "image_config.json").open(
        "w", encoding="utf-8"
    ) as file:
        json.dump(image_config, file, indent=2)

    print("Image feature extraction completed.")


## Image model selection


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import joblib
import numpy as np

loaded_image_features = {}

for split_name, split_df in [
    ("train", train_df),
    ("validation", val_df)
]:
    with np.load(
        FEATURE_DIR / f"image_{split_name}.npz",
        allow_pickle=False
    ) as saved:
        assert np.array_equal(
            saved["pet_ids"],
            split_df["PetID"].astype(str).to_numpy()
        )

        loaded_image_features[split_name] = (
            saved["embeddings"].copy()
        )

X_image_train = loaded_image_features["train"]
X_image_val = loaded_image_features["validation"]

y_train = train_df["AdoptionSpeed"].to_numpy()
y_val = val_df["AdoptionSpeed"].to_numpy()

image_search_rows = []
best_image_score = float("-inf")
image_model = None

for c_value in [0.1, 1.0, 10.0]:
    for class_weight in [None, "balanced"]:
        candidate = Pipeline([
            ("scaler", StandardScaler()),
            ("classifier", LogisticRegression(
                C=c_value,
                class_weight=class_weight,
                max_iter=3000,
                random_state=SEED
            ))
        ])

        candidate.fit(X_image_train, y_train)

        train_scores = evaluate_predictions(
            y_train,
            candidate.predict(X_image_train)
        )

        val_scores = evaluate_predictions(
            y_val,
            candidate.predict(X_image_val)
        )

        image_search_rows.append({
            "C": c_value,
            "ClassWeight": (
                "None" if class_weight is None else class_weight
            ),
            "TrainMacroF1": train_scores["MacroF1"],
            "ValidationMacroF1": val_scores["MacroF1"],
            "ValidationWeightedF1": val_scores["WeightedF1"],
            "ValidationQWK": val_scores["QWK"]
        })

        if val_scores["MacroF1"] > best_image_score:
            best_image_score = val_scores["MacroF1"]
            image_model = candidate

image_search_results = pd.DataFrame(
    image_search_rows
).sort_values(
    "ValidationMacroF1",
    ascending=False
).reset_index(drop=True)

joblib.dump(image_model, MODEL_DIR / "image_best.joblib")

image_search_results.to_csv(
    OUTPUT_DIR / "image_parameter_search.csv",
    index=False
)

display(image_search_results.round(4))
print("Best image model saved.")

## Early fusion


In [ ]:
from scipy import sparse
from sklearn.base import clone

tabular_model = joblib.load(MODEL_DIR / "tabular_best.joblib")

fusion_tabular_preprocessor = clone(
    tabular_model.named_steps["preprocessor"]
)

X_tabular_train = fusion_tabular_preprocessor.fit_transform(
    train_df[tabular_columns]
)

X_tabular_val = fusion_tabular_preprocessor.transform(
    val_df[tabular_columns]
)

fusion_image_scaler = StandardScaler()
fusion_text_scaler = StandardScaler()

X_image_train_scaled = fusion_image_scaler.fit_transform(
    X_image_train
)

X_image_val_scaled = fusion_image_scaler.transform(
    X_image_val
)

X_text_train_scaled = fusion_text_scaler.fit_transform(
    X_text_train
)

X_text_val_scaled = fusion_text_scaler.transform(
    X_text_val
)

X_early_train = sparse.hstack([
    sparse.csr_matrix(X_tabular_train),
    sparse.csr_matrix(X_image_train_scaled),
    sparse.csr_matrix(X_text_train_scaled)
], format="csr")

X_early_val = sparse.hstack([
    sparse.csr_matrix(X_tabular_val),
    sparse.csr_matrix(X_image_val_scaled),
    sparse.csr_matrix(X_text_val_scaled)
], format="csr")

early_search_rows = []
best_early_score = float("-inf")
early_model = None

for c_value in [0.01, 0.1, 1.0]:
    for class_weight in [None, "balanced"]:
        candidate = LogisticRegression(
            C=c_value,
            class_weight=class_weight,
            max_iter=3000,
            random_state=SEED
        )

        candidate.fit(X_early_train, y_train)

        train_scores = evaluate_predictions(
            y_train,
            candidate.predict(X_early_train)
        )

        val_scores = evaluate_predictions(
            y_val,
            candidate.predict(X_early_val)
        )

        early_search_rows.append({
            "C": c_value,
            "ClassWeight": (
                "None" if class_weight is None else class_weight
            ),
            "TrainMacroF1": train_scores["MacroF1"],
            "ValidationMacroF1": val_scores["MacroF1"],
            "ValidationWeightedF1": val_scores["WeightedF1"],
            "ValidationQWK": val_scores["QWK"]
        })

        if val_scores["MacroF1"] > best_early_score:
            best_early_score = val_scores["MacroF1"]
            early_model = candidate

early_search_results = pd.DataFrame(
    early_search_rows
).sort_values(
    "ValidationMacroF1",
    ascending=False
).reset_index(drop=True)

early_bundle = {
    "classifier": early_model,
    "tabular_preprocessor": fusion_tabular_preprocessor,
    "image_scaler": fusion_image_scaler,
    "text_scaler": fusion_text_scaler,
    "tabular_columns": tabular_columns,
    "feature_order": ["tabular", "image", "text"]
}

joblib.dump(
    early_bundle,
    MODEL_DIR / "early_fusion_best.joblib"
)

early_search_results.to_csv(
    OUTPUT_DIR / "early_fusion_parameter_search.csv",
    index=False
)

print("Training feature shape:", X_early_train.shape)
print("Validation feature shape:", X_early_val.shape)

display(early_search_results.round(4))
print("Best early fusion model saved.")

## Late fusion


In [ ]:
import json
import numpy as np

tabular_model = joblib.load(MODEL_DIR / "tabular_best.joblib")
image_model = joblib.load(MODEL_DIR / "image_best.joblib")
text_model = joblib.load(MODEL_DIR / "text_best.joblib")

classes = tabular_model.classes_

assert np.array_equal(classes, image_model.classes_)
assert np.array_equal(classes, text_model.classes_)

val_tabular_probs = tabular_model.predict_proba(
    val_df[tabular_columns]
)

val_image_probs = image_model.predict_proba(X_image_val)
val_text_probs = text_model.predict_proba(X_text_val)

equal_probs = (
    val_tabular_probs + val_image_probs + val_text_probs
) / 3

equal_predictions = classes[equal_probs.argmax(axis=1)]

equal_results = pd.DataFrame([{
    "Method": "Equal-weight late fusion",
    **evaluate_predictions(y_val, equal_predictions)
}])

late_search_rows = []
best_late_score = float("-inf")
best_late_weights = None

for tabular_step in range(11):
    for image_step in range(11 - tabular_step):
        text_step = 10 - tabular_step - image_step

        weights = np.array([
            tabular_step,
            image_step,
            text_step
        ], dtype=float) / 10

        probabilities = (
            weights[0] * val_tabular_probs
            + weights[1] * val_image_probs
            + weights[2] * val_text_probs
        )

        predictions = classes[probabilities.argmax(axis=1)]
        scores = evaluate_predictions(y_val, predictions)

        late_search_rows.append({
            "TabularWeight": weights[0],
            "ImageWeight": weights[1],
            "TextWeight": weights[2],
            **scores
        })

        if scores["MacroF1"] > best_late_score:
            best_late_score = scores["MacroF1"]
            best_late_weights = weights.copy()

late_search_results = pd.DataFrame(
    late_search_rows
).sort_values(
    "MacroF1",
    ascending=False,
    kind="stable"
).reset_index(drop=True)

late_config = {
    "weight_order": ["tabular", "image", "text"],
    "weights": best_late_weights.tolist(),
    "selection_metric": "validation_macro_f1",
    "grid_step": 0.1
}

with (MODEL_DIR / "late_fusion_weights.json").open(
    "w", encoding="utf-8"
) as file:
    json.dump(late_config, file, indent=2)

equal_results.to_csv(
    OUTPUT_DIR / "late_fusion_equal_validation.csv",
    index=False
)

late_search_results.to_csv(
    OUTPUT_DIR / "late_fusion_weight_search.csv",
    index=False
)

print("Equal-weight validation results:")
display(equal_results.round(4))

print("Top validation weight combinations:")
display(late_search_results.head(10).round(4))

print("Best weights:", late_config)
print("Late fusion weights and results saved.")

## Final test evaluation


In [ ]:
from scipy import sparse
from sklearn.dummy import DummyClassifier

test_features = {}

for modality in ["image", "text"]:
    with np.load(
        FEATURE_DIR / f"{modality}_test.npz",
        allow_pickle=False
    ) as saved:
        assert np.array_equal(
            saved["pet_ids"],
            test_df["PetID"].astype(str).to_numpy()
        )
        test_features[modality] = saved["embeddings"].copy()

X_image_test = test_features["image"]
X_text_test = test_features["text"]
y_test = test_df["AdoptionSpeed"].to_numpy()

early_bundle = joblib.load(
    MODEL_DIR / "early_fusion_best.joblib"
)

with (MODEL_DIR / "late_fusion_weights.json").open(
    encoding="utf-8"
) as file:
    late_config = json.load(file)

weights = np.asarray(late_config["weights"])

X_early_test = sparse.hstack([
    sparse.csr_matrix(
        early_bundle["tabular_preprocessor"].transform(
            test_df[early_bundle["tabular_columns"]]
        )
    ),
    sparse.csr_matrix(
        early_bundle["image_scaler"].transform(X_image_test)
    ),
    sparse.csr_matrix(
        early_bundle["text_scaler"].transform(X_text_test)
    )
], format="csr")

test_tabular_probs = tabular_model.predict_proba(
    test_df[tabular_columns]
)
test_image_probs = image_model.predict_proba(X_image_test)
test_text_probs = text_model.predict_proba(X_text_test)

test_late_probs = (
    weights[0] * test_tabular_probs
    + weights[1] * test_image_probs
    + weights[2] * test_text_probs
)

dummy_model = DummyClassifier(strategy="most_frequent")
dummy_model.fit(np.zeros((len(y_train), 1)), y_train)

test_predictions = {
    "Majority baseline": dummy_model.predict(
        np.zeros((len(y_test), 1))
    ),
    "Tabular": tabular_model.predict(
        test_df[tabular_columns]
    ),
    "Image": image_model.predict(X_image_test),
    "Text": text_model.predict(X_text_test),
    "Early Fusion": early_bundle["classifier"].predict(
        X_early_test
    ),
    "Late Fusion": classes[test_late_probs.argmax(axis=1)]
}

final_results = pd.DataFrame([
    {
        "Model": name,
        **evaluate_predictions(y_test, predictions)
    }
    for name, predictions in test_predictions.items()
])

prediction_table = pd.DataFrame({
    "PetID": test_df["PetID"].astype(str),
    "Actual": y_test,
    **test_predictions
})

final_results.to_csv(
    OUTPUT_DIR / "final_test_results.csv",
    index=False
)

prediction_table.to_csv(
    OUTPUT_DIR / "test_predictions.csv",
    index=False
)

display(final_results.round(4))
print("Final test results and predictions saved.")

## Class-level analysis


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import (
    classification_report,
    ConfusionMatrixDisplay
)

FIGURE_DIR = OUTPUT_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

prediction_table = pd.read_csv(
    OUTPUT_DIR / "test_predictions.csv"
)

labels = [0, 1, 2, 3, 4]
fusion_names = ["Early Fusion", "Late Fusion"]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for axis, model_name in zip(axes, fusion_names):
    ConfusionMatrixDisplay.from_predictions(
        prediction_table["Actual"],
        prediction_table[model_name],
        labels=labels,
        normalize="true",
        display_labels=labels,
        cmap="Blues",
        values_format=".2f",
        colorbar=False,
        ax=axis
    )

    axis.set_title(model_name)
    axis.set_xlabel("Predicted class")
    axis.set_ylabel("True class")

fig.suptitle("Row-normalized Test Confusion Matrices")
fig.tight_layout()

fig.savefig(
    FIGURE_DIR / "fusion_confusion_matrices.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

for model_name in fusion_names:
    report = classification_report(
        prediction_table["Actual"],
        prediction_table[model_name],
        labels=labels,
        output_dict=True,
        zero_division=0
    )

    report_df = pd.DataFrame(report).transpose()

    report_df.to_csv(
        OUTPUT_DIR
        / f"{model_name.lower().replace(' ', '_')}_classification_report.csv"
    )

    print(f"\n{model_name}:")
    display(
        report_df.loc[
            ["0", "1", "2", "3", "4", "macro avg", "weighted avg"],
            ["precision", "recall", "f1-score", "support"]
        ].round(4)
    )

print("Confusion matrices and classification reports saved.")

## Overfitting assessment


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

result_files = {
    "Tabular": "tabular_parameter_search.csv",
    "Image": "image_parameter_search.csv",
    "Text": "text_parameter_search.csv",
    "Early Fusion": "early_fusion_parameter_search.csv"
}

comparison_rows = []

for model_name, filename in result_files.items():
    search_df = pd.read_csv(OUTPUT_DIR / filename)
    best_row = search_df.iloc[0]

    comparison_rows.append({
        "Model": model_name,
        "TrainMacroF1": best_row["TrainMacroF1"],
        "ValidationMacroF1": best_row["ValidationMacroF1"],
        "Gap": (
            best_row["TrainMacroF1"]
            - best_row["ValidationMacroF1"]
        )
    })

overfitting_summary = pd.DataFrame(comparison_rows)

overfitting_summary.to_csv(
    OUTPUT_DIR / "overfitting_summary.csv",
    index=False
)

fig, ax = plt.subplots(figsize=(9, 5))

overfitting_summary.set_index("Model")[
    ["TrainMacroF1", "ValidationMacroF1"]
].plot.bar(
    ax=ax,
    color=["#4477AA", "#EEAA33"],
    rot=0
)

ax.set_title("Training and Validation Performance")
ax.set_xlabel("Model")
ax.set_ylabel("Macro-F1")
ax.set_ylim(0, 0.65)
ax.legend(["Training", "Validation"])
ax.grid(axis="y", alpha=0.3)

fig.tight_layout()
fig.savefig(
    FIGURE_DIR / "training_validation_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
display(overfitting_summary.round(4))

## Environment and experiment records


In [ ]:
import sys
import json
import platform
import importlib.metadata
import numpy as np

package_names = [
    "numpy",
    "pandas",
    "scipy",
    "scikit-learn",
    "torch",
    "torchvision",
    "sentence-transformers",
    "transformers",
    "Pillow",
    "matplotlib",
    "joblib"
]

package_versions = {}

for package_name in package_names:
    try:
        package_versions[package_name] = (
            importlib.metadata.version(package_name)
        )
    except importlib.metadata.PackageNotFoundError:
        package_versions[package_name] = "Not installed"

experiment_config = {
    "random_seed": 42,
    "total_records": 14993,
    "final_records": 14640,
    "train_records": 10248,
    "validation_records": 2196,
    "test_records": 2196,
    "split_method": "Stratified random split by AdoptionSpeed",
    "split_ratio": [0.70, 0.15, 0.15],
    "image_selection": "First image per pet",
    "primary_selection_metric": "Validation Macro-F1",
    "test_used_for_parameter_selection": False,
    "python_version": sys.version,
    "platform": platform.platform(),
    "package_versions": package_versions
}

with (OUTPUT_DIR / "experiment_config.json").open(
    "w", encoding="utf-8"
) as file:
    json.dump(experiment_config, file, indent=2)

with (PROJECT_DIR / "requirements.txt").open(
    "w", encoding="utf-8"
) as file:
    for package_name, version in package_versions.items():
        if version != "Not installed":
            file.write(f"{package_name}=={version}\n")

for split_name, split_df in [
    ("train", train_df),
    ("validation", val_df),
    ("test", test_df)
]:
    split_df[["PetID"]].to_csv(
        SPLIT_DIR / f"{split_name}_ids.csv",
        index=False
    )

print("Experiment configuration saved.")
print("Package versions saved.")
print("Split PetIDs saved.")

display(
    pd.DataFrame(
        package_versions.items(),
        columns=["Package", "Version"]
    )
)